# Amaliy mashgʻulot 16. Katta til modellari: matn generatsiyasi va soʻrovlarni loyihalash

16-laboratoriyada siz harorat, top-k va top-p slayderlarini surgansiz, zero-shot, few-shot va chain-of-thought soʻrovlarini solishtirgansiz hamda bir nechta mulohazalar ovozi (self-consistency) aniqlikni qanday oshirishini koʻrgansiz. Bu yerda xuddi shuni kodda bajaramiz:

1. greedy, harorat, top-k va top-p ni noldan yozamiz va ularni xossalari bilan tekshiramiz;
2. bigramm til modelini oʻrgatamiz va generatsiya strategiyalarini solishtiramiz: xilma-xillik va sifat;
3. zero-shot / few-shot / CoT soʻrov shablonlarini yigʻamiz, javobni ajratib olish va tekshirishni oʻrganamiz;
4. self-consistency ovoz berish aniqligini binomial formula va simulyatsiya bilan hisoblaymiz;
5. Colab da bularning barchasini kichik haqiqiy modelda sinab koʻramiz.

*«NLP algoritmlari» kursining 16-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Dekodlash: logitlardan keyingi tokenga

Har qadamda til modeli **logitlar** vektori $z$ ni beradi — lugʻatdagi har bir token uchun bitta son. Keyingi tokenni qanday tanlash mumkin?

- **greedy** (ochkoʻz): $\arg\max_i z_i$ — deterministik, lekin bir xil;
- **harorat**: $p_i = \dfrac{e^{z_i/T}}{\sum_j e^{z_j/T}}$. $T<1$ da taqsimot keskinlashadi, $T>1$ da tekislanadi, $T\to 0$ da greedy hosil boʻladi;
- **top-k**: eng ehtimolli $k$ ta tokenni qoldirib, qayta normallash;
- **top-p** (nucleus): ehtimollar yigʻindisi $\ge p$ boʻlgan eng ehtimolli tokenlarning eng kichik toʻplamini qoldirib, qayta normallash. «Yadro» hajmi modelning ishonchiga qarab oʻzgaradi.

«Nonushtada men odatda …» gapining davomi uchun oʻyinchoq logitlar vektorini olamiz.

In [ ]:
import numpy as np

context = "Nonushtada men odatda"
vocab = ["choy", "qahva", "sut", "suv", "sharbat", "kakao", "qatiq", "ayron", "kompot", "limonad"]
logits = np.array([3.2, 2.9, 1.8, 1.5, 1.2, 0.6, 0.3, -0.2, -0.5, -1.0])   # keyingi soʻz uchun model chiqishi


def softmax_t(logits, T=1.0):
    # T haroratli softmax
    z = np.asarray(logits, dtype=float) / T
    e = np.exp(z - z.max())
    return e / e.sum()

def entropy(p):
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum()) + 0.0       # + 0.0 greedy dagi «-0.00» ni yoʻqotadi

def greedy(p):
    q = np.zeros_like(p, dtype=float)
    q[np.argmax(p)] = 1.0
    return q

def top_k_filter(p, k):
    # eng ehtimolli k ta tokenni qoldirish
    keep = np.argsort(-p, kind="stable")[:k]
    q = np.zeros_like(p)
    q[keep] = p[keep]
    return q / q.sum()

def top_p_filter(p, top_p):
    # ehtimollar yigʻindisi >= top_p boʻlgan eng kichik tokenlar toʻplami
    order = np.argsort(-p, kind="stable")
    cum = np.cumsum(p[order])
    n = min(int(np.searchsorted(cum, top_p)) + 1, len(p))
    q = np.zeros_like(p)
    q[order[:n]] = p[order[:n]]
    return q / q.sum()

def decode_dist(logits, T=1.0, k=None, top_p=None):
    # tartib Hugging Face dagi kabi: harorat -> top-k -> top-p; T=0 greedy degani
    if T == 0:
        return greedy(np.asarray(logits, dtype=float))
    q = softmax_t(logits, T)
    if k is not None:
        q = top_k_filter(q, k)
    if top_p is not None:
        q = top_p_filter(q, top_p)
    return q

settings = {"T=1": dict(T=1.0), "T=0.5": dict(T=0.5), "T=2": dict(T=2.0),
            "top-k=3": dict(k=3), "top-p=0.9": dict(top_p=0.9), "greedy": dict(T=0)}
dists = {name: decode_dist(logits, **kw) for name, kw in settings.items()}

print(context, "...\n")
print(f"{'':<10}{'logit':>6}" + "".join(f"{n:>11}" for n in dists))
for i, w in enumerate(vocab):
    print(f"{w:<10}{logits[i]:6.1f}" + "".join(f"{d[i]:11.3f}" for d in dists.values()))
print(f"{'entropiya, bit':<16}" + "".join(f"{entropy(d):11.2f}" for d in dists.values()))

Amalga oshirishni har doim bajarilishi kerak boʻlgan xossalar bilan tekshiramiz: $k=1$ boʻlgan top-k va juda kichik $p$ li top-p greedy bilan bir xil; $T\to 0$ da greedy, $T\to\infty$ da tekis taqsimot hosil boʻladi; harorat tokenlar tartibini oʻzgartirmaydi. Soʻng sempling (tasodifiy tanlash) taqsimotni haqiqatan takrorlashiga ishonch hosil qilamiz.


Grafikda kulrang ustunlar — $T=1$ dagi dastlabki taqsimot, koʻklar — oʻzgartirilgandan keyingisi. top-k va top-p kesib tashlagan tokenlar kulrangligicha qoladi: ularning ehtimoli nolga teng boʻldi, qolganlariniki esa oshdi.

In [ ]:
p1 = softmax_t(logits)
assert all(np.isclose(d.sum(), 1.0) for d in dists.values())        # barcha taqsimotlar yigʻindisi 1
assert np.allclose(top_k_filter(p1, 1), greedy(p1))                 # k=1 li top-k — bu greedy
assert np.allclose(top_p_filter(p1, 1e-9), greedy(p1))              # juda kichik p li top-p ham greedy
assert np.allclose(top_k_filter(p1, len(p1)), p1)                   # k = |V| va p = 1 hech narsani oʻzgartirmaydi
assert np.allclose(top_p_filter(p1, 1.0), p1)
assert np.allclose(softmax_t(logits, 0.01), greedy(p1), atol=1e-6)  # T -> 0: greedy
assert np.allclose(softmax_t(logits, 1000), 1 / len(p1), atol=1e-2) # T -> cheksizlik: tekis taqsimot
assert all(np.argmax(softmax_t(logits, T)) == np.argmax(logits) for T in [0.1, 1, 10])  # harorat eng ehtimolli tokenni oʻzgartirmaydi
print("Barcha tekshiruvlar oʻtdi.\n")

for T in [0.25, 0.5, 1, 2, 4]:
    q = softmax_t(logits, T)
    n_core = int((top_p_filter(q, 0.9) > 0).sum())
    print(f"T={T:<5} H={entropy(q):.2f} bit   p({vocab[0]})={q[0]:.2f}   top-p 0.9 yadrosi, token = {n_core}")

rng = np.random.default_rng(0)
q = dists["top-p=0.9"]
draws = rng.choice(len(vocab), size=20000, p=q)
freq = np.bincount(draws, minlength=len(vocab)) / len(draws)
print("\ntop-p=0.9 sempling, 20000 marta: ehtimol va chastota")
for w, a, b in zip(vocab, q, freq):
    print(f"   {w:<10} p = {a:.3f}   chastota = {b:.3f}")


import matplotlib.pyplot as plt

show = ["T=1", "T=0.5", "T=2", "top-k=3", "top-p=0.9"]
x = np.arange(len(vocab))
fig, axes = plt.subplots(1, len(show), figsize=(15, 3.2), sharey=True)
for ax, name in zip(axes, show):
    d = dists[name]
    ax.bar(x, p1, color="#DDDDDD", label="T=1")                       # kulrang — T=1 dagi dastlabki taqsimot
    ax.bar(x, d, width=0.55, color=["#2B55C9" if v > 0 else "#DDDDDD" for v in d], label=name)
    ax.set_title(name)
    ax.set_xticks(x)
    ax.set_xticklabels(vocab, rotation=70, fontsize=8)
axes[0].set_ylabel("ehtimol")
fig.suptitle("Dekodlash strategiyalari: «" + context + " …»")
plt.tight_layout()
plt.show()

### Haqiqiy modelda

`Qwen/Qwen2.5-0.5B-Instruct` modelining (0,5 mlrd parametr, fayli ≈1 GB) shu gap uchun haqiqiy logitlarini olib, ularga oʻz funksiyalarimizni qoʻllaymiz. Model lugʻati — 150 mingga yaqin token. Turli haroratlarda ulardan nechtasi top-p «yadrosi»ga tushishini koʻring.

Oxirida bir xil gap rus, oʻzbek va ingliz tillarida nechta tokenga boʻlinishini solishtiramiz. Tokenizator oʻqitilgan maʼlumotlarda kam boʻlgan tillar mayda boʻlaklarga boʻlinadi (2- va 17-mavzularga qarang): bunday tildagi soʻrov uzunroq va qimmatroq boʻladi. Kichik modellar oʻzbek tilida ham ancha zaif. Katak faqat internetli Colab da ishlaydi.

In [ ]:
# Colab da transformers va torch allaqachon oʻrnatilgan; ≈1 GB model bir marta yuklanadi
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = "Qwen/Qwen2.5-0.5B-Instruct"                     # 0,5 mlrd parametr
    tok = AutoTokenizer.from_pretrained(name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = "auto" if device == "cuda" else torch.float32     # CPU da float32 bfloat16 dan tezroq
    llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)

    ids = tok(context, return_tensors="pt").to(device)
    with torch.no_grad():
        real_logits = llm(**ids).logits[0, -1].float().cpu().numpy()   # keyingi token uchun logitlar
    print("model lugʻati hajmi:", real_logits.shape[0])
    p_real = softmax_t(real_logits)
    for i in np.argsort(-p_real)[:10]:
        print(f"   {tok.decode([int(i)])!r:<14} {p_real[i]:.3f}")
    for T in [0.5, 1.0, 1.5]:
        q = softmax_t(real_logits, T)
        n_core = int((top_p_filter(q, 0.9) > 0).sum())
        print(f"T={T}: H = {entropy(q):.2f} bit, top-p 0.9 yadrosi, token = {n_core}")

    # bir xil gap uch tilda
    same = {"ru": "Завтра мы поедем на поезде из Ташкента в Самарканд.",
            "uz": "Ertaga biz poyezdda Toshkentdan Samarqandga boramiz.",
            "en": "Tomorrow we will take the train from Tashkent to Samarkand."}
    print()
    for lang, s in same.items():
        pieces = [tok.decode([t]) for t in tok(s)["input_ids"]]     # boʻlak harfning yarmi boʻlishi mumkin (�): tokenlar — UTF-8 baytlari
        print(f"{lang}: {len(s.split())} soʻz -> {len(pieces)} token: {pieces}")
except Exception as e:
    print("Modelni ishga tushirib boʻlmadi (Colab da internet kerak):", e)

## 2. Bigramm model bilan matn generatsiyasi

Haqiqiy LLM — xuddi shu «logitlar → dekodlash → token» sxemasining koʻp marta takrorlanishi. Eng oddiy til modelini — bigramm modelni olamiz (4-mavzu): $P(w_i\mid w_{i-1})=\dfrac{c(w_{i-1},w_i)}{c(w_{i-1})}$. Logitlar — 30 ta gapdan iborat korpusda $w_{i-1}$ dan keyin uchragan soʻzlar uchun $\log P(w\mid w_{i-1})$. Ehtimollarimizni `nltk.lm.MLE` bilan solishtiramiz.

In [ ]:
corpus = [
    "biz ertaga poyezdda samarqandga boramiz", "biz ertaga mashinada buxoroga boramiz",
    "men ertaga poyezdda toshkentga boraman", "samarqandga poyezd ertalab joʻnaydi",
    "buxoroga poyezd kechqurun joʻnaydi", "ertalab biz choy ichamiz", "kechqurun biz sutli choy ichamiz",
    "samarqandda biz registonni koʻramiz", "buxoroda biz eski shaharni koʻramiz", "eski shahar juda chiroyli",
    "registon juda chiroyli", "men koʻk choyni yaxshi koʻraman", "men eski shaharni yaxshi koʻraman",
    "biz poyezdga chipta oldik", "poyezd chiptasi qimmat emas", "samarqandda havo issiq",
    "buxoroda havo juda issiq", "kechqurun havo salqin boʻladi", "ertalab havo salqin boʻladi",
    "men samarqanddan non olib keldim", "biz buxorodan sovgʻa olib keldik", "samarqand noni juda mazali",
    "bu choy juda mazali", "onam ertalab non yopadi", "onam kechqurun osh pishiradi",
    "biz kechqurun osh yeymiz", "men oshni yaxshi koʻraman", "toshkentda metro bor",
    "toshkentda biz metroda yuramiz", "poyezd toshkentga kechqurun yetib keladi",
]
PREV_DEMO = "biz"


import re
from collections import Counter, defaultdict

def tokenize(s):
    return re.findall(r"[^\W\d_]+", s.lower())

bigrams = defaultdict(Counter)
for s in corpus:
    toks = ["<s>"] + tokenize(s) + ["</s>"]
    for a, b in zip(toks, toks[1:]):
        bigrams[a][b] += 1

def next_logits(prev):
    # logitlar = prev dan keyin uchragan soʻzlar uchun log P(w | prev)
    words = sorted(bigrams[prev])
    counts = np.array([bigrams[prev][w] for w in words], dtype=float)
    return words, np.log(counts / counts.sum())

words, lp = next_logits(PREV_DEMO)
print(f"P(w | {PREV_DEMO}):")
for T in [1.0, 0.5, 2.0]:
    q = decode_dist(lp, T)
    print(f"  T={T}: " + ", ".join(f"{w} {x:.2f}" for w, x in sorted(zip(words, q), key=lambda t: -t[1])))

# nltk.lm bilan solishtirish (silliqlashsiz, MLE)
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline
train_ng, lm_vocab = padded_everygram_pipeline(2, [tokenize(s) for s in corpus])
lm = MLE(2)
lm.fit(train_ng, lm_vocab)
diff = max(abs(np.exp(l) - lm.score(w, [prev]))
           for prev in list(bigrams) for w, l in zip(*next_logits(prev)))
print(f"\nlugʻat: {len(lm.vocab) - 1}, xil bigramm: {sum(len(c) for c in bigrams.values())}")
print(f"max |bizniki - nltk| = {diff:.2e}")

Turli strategiyalar bilan gaplar generatsiya qilamiz. Greedy har doim bir xil gapni beradi; past harorat tez-tez uchraydigan yoʻllarni ushlaydi; yuqori harorat gʻayrioddiy birikmalarni yigʻadi.

In [ ]:
def generate(T=1.0, k=None, top_p=None, rng=None, max_len=12):
    # soʻzma-soʻz generatsiya; matn va T=1 model boʻyicha oʻrtacha log P ni qaytaradi
    prev, out, logps = "<s>", [], []
    while len(out) < max_len:
        words, lp = next_logits(prev)
        q = decode_dist(lp, T, k, top_p)
        i = int(np.argmax(q)) if T == 0 else int(rng.choice(len(words), p=q))
        logps.append(lp[i])
        if words[i] == "</s>":
            break
        out.append(words[i])
        prev = words[i]
    return " ".join(out), float(np.mean(logps))

rng = np.random.default_rng(1)
print("greedy:      ", generate(T=0)[0])
for name, kw in [("T=0.5", dict(T=0.5)), ("T=1", dict(T=1.0)), ("T=3", dict(T=3.0)),
                 ("top-k=2", dict(k=2)), ("top-p=0.6", dict(top_p=0.6))]:
    for _ in range(3):
        print(f"{name:<12}", generate(rng=rng, **kw)[0])
    print()

### Xilma-xillik va sifat

Har bir strategiya uchun 300 tadan gap generatsiya qilib, quyidagilarni hisoblaymiz:

- **turli** — 300 ta gap orasida bir-biridan farq qiladigan gaplar ulushi (xilma-xillik);
- **yangi** — korpusda yoʻq gaplar ulushi (model takrorlamaydi, «toʻqiydi»);
- **oʻrtacha log P** — dastlabki model ($T=1$) boʻyicha har bir soʻzga: 0 ga qanchalik yaqin boʻlsa, matn model uchun shunchalik odatiy.

Xilma-xillik greedy dan $Tpprox 1$ gacha tez oʻsadi, keyin deyarli oʻzgarmaydi: kichik bigramm modelda mumkin boʻlgan gaplar oz. «Odatiylik» (oʻrtacha log P) $T$ oshishi bilan kamayadi. top-k va top-p kam uchraydigan davomlarni kesib tashlaydi, shuning uchun $T=1$ dagi oddiy semplingga qaraganda odatiyroq matn beradi. Bigramm modelda har bir qoʻshni soʻzlar jufti korpusda boʻlgan, shuning uchun mahalliy jihatdan hammasi silliq, lekin model faqat bitta oldingi soʻzni koʻradi: moslashuv va butun gapning maʼnosi yoʻqoladi («men … non yopadi» — shaxs-son qoʻshimchasi mos emas, «onam ertalab havo issiq»).

In [ ]:
train_set = {" ".join(tokenize(s)) for s in corpus}

def evaluate(n=300, **kw):
    sents = [generate(rng=rng, **kw) for _ in range(n)]
    distinct = len({s for s, _ in sents}) / n                       # n ta gap orasida turli gaplar ulushi
    novel = np.mean([s not in train_set for s, _ in sents])
    return distinct, novel, np.mean([l for _, l in sents])

rng = np.random.default_rng(2)
rows = []
print(f"{'':<12}{'turli':>12}{'yangi':>10}{'oʻrtacha log P':>17}")
for name, kw in [("greedy", dict(T=0)), ("T=0.3", dict(T=0.3)), ("T=0.5", dict(T=0.5)),
                 ("T=1", dict(T=1.0)), ("T=1.5", dict(T=1.5)), ("T=2", dict(T=2.0)), ("T=3", dict(T=3.0)),
                 ("top-k=2", dict(k=2)), ("top-p=0.6", dict(top_p=0.6))]:
    dist, nov, alp = evaluate(**kw)
    rows.append((name, dist, nov, alp))
    print(f"{name:<12}{dist:12.2f}{nov:10.2f}{alp:17.2f}")

temp_rows = [r for r in rows if r[0].startswith("T=")]
Ts = [float(r[0][2:]) for r in temp_rows]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(Ts, [r[1] for r in temp_rows], "o-", label="turli")
ax[0].plot(Ts, [r[2] for r in temp_rows], "s-", label="yangi")
ax[0].set_xlabel("harorat T"); ax[0].set_title("Xilma-xillik"); ax[0].legend()
ax[1].plot(Ts, [r[3] for r in temp_rows], "o-", color="#D9822B")
ax[1].set_xlabel("harorat T"); ax[1].set_title("Odatiylik: soʻzga oʻrtacha log P")
plt.tight_layout()
plt.show()

## 3. Soʻrovlarni loyihalash (prompt engineering)

Soʻrov (prompt) — generatsiya boshlanadigan oddiy satr. Javob uning koʻrinishiga kuchli bogʻliq:

- **zero-shot** — faqat koʻrsatma va savol;
- **few-shot** — savoldan oldin bir nechta yechilgan misol (in-context learning: model vaznlari oʻzgarmaydi, u kontekstdan «oʻrganadi»);
- **chain-of-thought (CoT, fikrlash zanjiri)** — misollarda oraliq qadamlar koʻrsatiladi yoki soʻrovga «Keling, qadamma-qadam fikr yuritamiz» iborasi qoʻshiladi (zero-shot CoT).

Nega CoT koʻp qadamli arifmetikada yordam beradi? Model har bir token uchun bir xil hajmda hisoblash bajaradi. Agar javobni darhol berish kerak boʻlsa, barcha qadamlarni bitta tokenda «xayolan» bajarishga toʻgʻri keladi. Qadamlar yozilsa, har bir qadam oddiy amal boʻladi, uning natijasi esa kontekstda turadi va keyin ishlatiladi.

Shablonlarni Python funksiyasi bilan yigʻamiz. Masalalarning toʻgʻri javoblarini kod bilan hisoblaymiz. Narxga eʼtibor bering: few-shot va CoT ancha uzun, pul esa tokenlar uchun toʻlanadi.

In [ ]:
problems = [
    "Poyezdda 8 ta vagon bor, har birida 50 ta oʻrin. 270 ta chipta sotildi. Nechta oʻrin boʻsh qoldi?",
    "Katta yoshlilar chiptasi 120 ming soʻm, bolalar chiptasi esa ikki baravar arzon. Oila 3 ta katta va 2 ta bolalar chiptasini sotib oldi. Oila necha ming soʻm toʻladi?",
    "Alisherda 5 ta daftar bor edi. U yana 4 tadan daftar solingan 3 ta qadoq sotib oldi va ukasiga 6 ta daftar sovgʻa qildi. Unda nechta daftar qoldi?",
    "Sinfda 36 nafar oʻquvchi bor. Ularning uchdan bir qismi Buxoroga sayohatga ketdi, qolganlari teng 4 guruhga boʻlindi. Har bir guruhda nechta oʻquvchi bor?",
    "Bozorda 1 kg olma 15 ming soʻm, 1 kg uzum esa 7 ming soʻm qimmatroq. 2 kg olma va 3 kg uzum necha ming soʻm turadi?",
]
# toʻgʻri javoblarni qoʻlda emas, kod bilan hisoblaymiz
gold = [8 * 50 - 270, 3 * 120 + 2 * (120 // 2), 5 + 3 * 4 - 6, (36 - 36 // 3) // 4, 2 * 15 + 3 * (15 + 7)]
print("gold:", gold)

INSTR = "Masalani yeching. Oxirida javobni «Javob: son» koʻrinishida yozing."
Q_LABEL, S_LABEL, A_LABEL = "Masala", "Yechim", "Javob"
COT_PHRASE = "Keling, qadamma-qadam fikr yuritamiz."
EXAMPLES = [   # few-shot uchun (masala, yechim qadamlari, javob)
    ("Savatda 12 ta olma bor edi. 5 tasi yeyildi, keyin yana 4 ta solindi. Savatda nechta olma bor?", "12 - 5 = 7, 7 + 4 = 11.", 11),
    ("Daftar 3 ming soʻm, ruchka 2 ming soʻm turadi. 4 ta daftar va 2 ta ruchka necha ming soʻm turadi?", "4 · 3 = 12, 2 · 2 = 4, 12 + 4 = 16.", 16),
]


MODES = ["zero-shot", "few-shot", "zero-shot CoT", "few-shot CoT"]

def make_prompt(question, mode):
    parts = [INSTR]
    if mode.startswith("few-shot"):
        for q, steps, a in EXAMPLES:
            if "CoT" in mode:
                parts.append(f"{Q_LABEL}: {q}\n{S_LABEL}: {steps} {A_LABEL}: {a}")
            else:
                parts.append(f"{Q_LABEL}: {q}\n{A_LABEL}: {a}")       # mulohazasiz few-shot: faqat javob
    if mode == "zero-shot CoT":
        tail = f"{S_LABEL}: {COT_PHRASE}"
    elif "CoT" in mode:
        tail = f"{S_LABEL}:"
    else:
        tail = f"{A_LABEL}:"
    parts.append(f"{Q_LABEL}: {question}\n{tail}")
    return "\n\n".join(parts)

for mode in MODES:
    print(f"==================== {mode} ====================")
    print(make_prompt(problems[1], mode))
    print()

print(f"{'':<15}{'belgi':>9}{'soʻz':>8}")
for mode in MODES:
    p = make_prompt(problems[1], mode)
    print(f"{mode:<15}{len(p):9d}{len(p.split()):8d}")

Quyida tasvir uchun qoʻlda yozilgan **tayyorlab qoʻyilgan javoblar** (bu model chiqishi emas). Ular kichik model uchun xos: mulohazasiz model koʻpincha sonni «taxmin qiladi», mulohaza bilan esa koʻproq toʻgʻri javob beradi, lekin arifmetikada xato qilishi mumkin.

Mulohazaning yana bir afzalligi bor: uni **tekshirish** mumkin. `check_steps` funksiyasi matndan «3 · 22 = 66» kabi tengliklarni topib, qayta hisoblaydi, `extract_answer` esa «Javob:» dan keyingi yakuniy sonni ajratib oladi.

In [ ]:
# TAYYORLAB QOʻYILGAN javoblar (tasvir uchun qoʻlda yozilgan, bu model chiqishi emas)
prepared = {
    (1, "zero-shot"): "Javob: 600",
    (1, "few-shot"): "Javob: 360",
    (1, "zero-shot CoT"): "Katta yoshlilar chiptalari: 3 · 120 = 360. Bolalar chiptasi ikki baravar arzon: 120 / 2 = 60. "
                          "Bolalar chiptalari: 2 · 60 = 120. Jami: 360 + 120 = 480. Javob: 480",
    (1, "few-shot CoT"): "3 · 120 = 360, 120 / 2 = 60, 2 · 60 = 120, 360 + 120 = 480. Javob: 480",
    (4, "zero-shot"): "Javob: 66",
    (4, "few-shot"): "Javob: 75",
    (4, "zero-shot CoT"): "Uzum 15 + 7 = 22 ming soʻm turadi. Olma: 2 · 15 = 30. Uzum: 3 · 22 = 56. "
                          "Jami: 30 + 56 = 86. Javob: 86",
    (4, "few-shot CoT"): "15 + 7 = 22, 2 · 15 = 30, 3 · 22 = 66, 30 + 66 = 96. Javob: 96",
}
ANSWER_RE = r"Javob[:\s*]*(-?\d+)"


def extract_answer(text):
    # «Javob:» dan keyingi son; boʻlmasa — matndagi oxirgi son
    m = re.findall(ANSWER_RE, text, flags=re.IGNORECASE)
    if m:
        return int(m[-1])
    nums = re.findall(r"-?\d+", text)
    return int(nums[-1]) if nums else None

OPS = {"+": lambda a, b: a + b, "-": lambda a, b: a - b, "·": lambda a, b: a * b,
       "*": lambda a, b: a * b, "×": lambda a, b: a * b, "/": lambda a, b: a / b}

def check_steps(text):
    # «a op b = c» tengliklarini topib, qayta hisoblaymiz
    bad = []
    for a, op, b, c in re.findall(r"(\d+)\s*([-+·*×/])\s*(\d+)\s*=\s*(\d+)", text):
        if OPS[op](int(a), int(b)) != int(c):
            bad.append(f"{a} {op} {b} = {c} (boʻlishi kerak {OPS[op](int(a), int(b)):g})")
    return bad

for (i, mode), text in prepared.items():
    pred = extract_answer(text)
    steps = check_steps(text)
    verdict = "OK " if pred == gold[i] else "XATO"
    print(f"masala {i}  {mode:<14} {verdict}  {A_LABEL.lower()}={pred:<4} gold={gold[i]:<4} "
          + (f"notoʻgʻri qadam: {steps}" if steps else ""))

### Haqiqiy modelda

`Qwen/Qwen2.5-0.5B-Instruct` beshta masalani toʻrt xil soʻrov bilan yechadi (greedy dekodlash, 256 tagacha yangi token). CPU da bu bir necha daqiqa, GPU da («Runtime → Change runtime type → T4») bir daqiqacha davom etadi. Model kichik: xatolar koʻp boʻladi, 5 ta masaladagi natija esa juda shovqinli, shuning uchun faqat hisobga emas, javoblarning oʻziga qarang.

**Muhim:** kichik modellar oʻzbek tilida rus va ingliz tillariga qaraganda ancha zaif: javob rus yoki ingliz tilida boʻlishi, «Javob:» formati buzilishi mumkin. Taqqoslash uchun `INSTR`, `Q_LABEL` va masalalarni rus yoki ingliz tiliga tarjima qilib, natijani solishtirib koʻring.

In [ ]:
# barcha masalalar × barcha soʻrov turlari; model hali yuklanmagan boʻlsa, yuklanadi
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if "llm" not in globals():
        name = "Qwen/Qwen2.5-0.5B-Instruct"
        tok = AutoTokenizer.from_pretrained(name)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = "auto" if device == "cuda" else torch.float32     # CPU da float32 bfloat16 dan tezroq
        llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)

    def ask(prompt, max_new_tokens=256, **gen):
        text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                       tokenize=False, add_generation_prompt=True)
        inputs = tok([text], return_tensors="pt").to(llm.device)
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id, **gen)
        return [tok.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True) for o in out]

    score = {m: 0 for m in MODES}
    for i, (q, g) in enumerate(zip(problems, gold)):
        for mode in MODES:
            answer = ask(make_prompt(q, mode), do_sample=False)[0]      # greedy dekodlash
            pred = extract_answer(answer)
            score[mode] += int(pred == g)
            print(f"{i} {mode:<14} pred={pred} gold={g}  " + answer.replace("\n", " ")[:110])
    print()
    for mode in MODES:
        print(f"{mode:<14} {score[mode]} / {len(problems)}")
except Exception as e:
    print("Modelni ishga tushirib boʻlmadi (Colab da internet kerak):", e)

## 4. Self-consistency: bir nechta mulohaza ovozi

Bitta greedy javob oʻrniga sempling ($T>0$) bilan $N$ ta mulohaza generatsiya qilamiz va eng koʻp uchragan yakuniy javobni tanlaymiz. Har bir mulohaza mustaqil ravishda $p$ ehtimol bilan toʻgʻri boʻlsin. Agar barcha notoʻgʻri javoblar bir xil boʻlsa (eng yomon holat: javobning faqat ikki varianti), toq $N$ uchun

$$P_{\text{maj}}(N,p)=\sum_{k=(N+1)/2}^{N}\binom{N}{k}\,p^k(1-p)^{N-k}.$$

$p>0.5$ da aniqlik $N$ bilan 1 ga qarab oʻsadi, $p<0.5$ da 0 ga qarab tushadi. Amalda notoʻgʻri javoblar odatda **har xil**, ularning ovozlari «tarqaladi» va nisbiy koʻpchilik ovozi $p<0.5$ da ham yordam beradi. Formulani simulyatsiya bilan tekshiramiz (durangni tasodifiy hal qilamiz).

Izoh: bitta modelning mulohazalari mustaqil emas (model bir xil xatolarni takrorlaydi), shuning uchun haqiqiy yutuq formuladagidan kamroq.


Katak oxirida — xuddi shu narsa «matnli» mulohaza yoʻllarida: har bir yoʻl «Javob: …» satri bilan tugaydi, javobni `extract_answer` bilan ajratib olamiz va ovoz beramiz. Yoʻllar modellashtirilgan: 0.6 ehtimol bilan toʻgʻri javob, aks holda odatiy xatolardan biri.

In [ ]:
from math import comb
from scipy.stats import binom

def p_majority(N, p):
    # koʻpchilik ovozi aniqligi, N toq, javobning ikki varianti
    return sum(comb(N, k) * p ** k * (1 - p) ** (N - k) for k in range((N + 1) // 2, N + 1))

assert np.isclose(p_majority(3, 0.6), 3 * 0.6 ** 2 * 0.4 + 0.6 ** 3)          # = 0.648
assert all(np.isclose(p_majority(N, 0.6), binom.sf((N - 1) // 2, N, 0.6)) for N in [1, 5, 11])

def simulate(N, p, m_wrong, trials=20000, rng=None):
    # har bir yoʻl p ehtimol bilan toʻgʻri, aks holda m_wrong ta xatodan biri (teng ehtimolli)
    probs = [p] + [(1 - p) / m_wrong] * m_wrong
    answers = rng.choice(m_wrong + 1, size=(trials, N), p=probs)             # 0 = toʻgʻri javob
    counts = (answers[:, :, None] == np.arange(m_wrong + 1)).sum(axis=1)
    winner = np.argmax(counts + 0.5 * rng.random(counts.shape), axis=1)     # durangni tasodifiy hal qilish
    return float(np.mean(winner == 0))

rng = np.random.default_rng(0)
Ns = [1, 3, 5, 7, 9, 11, 15, 21]
table = []
print(f"{'N':>3}{'p=0.6 formula':>18}{'simul.':>7}{'p=0.4 formula':>18}{'simul.':>7}{'p=0.4, 5 xil xato':>22}")
for N in Ns:
    row = (N, p_majority(N, 0.6), simulate(N, 0.6, 1, rng=rng), p_majority(N, 0.4),
           simulate(N, 0.4, 1, rng=rng), simulate(N, 0.4, 5, rng=rng))
    table.append(row)
    print(f"{row[0]:>3}{row[1]:>18.3f}{row[2]:>7.3f}{row[3]:>18.3f}{row[4]:>7.3f}{row[5]:>22.3f}")

table = np.array(table)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(table[:, 0], table[:, 1], "-", color="#2B55C9", label="p=0.6, formula")
ax.plot(table[:, 0], table[:, 2], "o", color="#2B55C9", label="p=0.6, simul.")
ax.plot(table[:, 0], table[:, 3], "-", color="#C0392B", label="p=0.4, formula")
ax.plot(table[:, 0], table[:, 4], "o", color="#C0392B", label="p=0.4, simul.")
ax.plot(table[:, 0], table[:, 5], "s--", color="#17907E", label="p=0.4, 5 xil xato (simul.)")
ax.axhline(0.5, color="grey", lw=0.5)
ax.set_xlabel("mulohazalar soni N")
ax.set_ylabel("ovoz berish aniqligi")
ax.set_title("Self-consistency")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


# 1-masala uchun modellashtirilgan mulohazalar (toʻgʻri javob 480)
wrong_answers = [600, 360, 540, 420]

def sample_path(rng, p=0.6):
    a = gold[1] if rng.random() < p else int(rng.choice(wrong_answers))
    return f"... {A_LABEL}: {a}"

rng = np.random.default_rng(5)
for N in [1, 5, 15]:
    paths = [sample_path(rng) for _ in range(N)]
    votes = Counter(extract_answer(t) for t in paths)
    winner = votes.most_common(1)[0][0]
    print(f"N={N:<3} ovozlar: {dict(votes)}  ->  {winner}  ({'OK' if winner == gold[1] else 'XATO'})")

### Haqiqiy modelda

`Qwen/Qwen2.5-0.5B-Instruct` ning bozor haqidagi masala uchun yettita mulohazasi (zero-shot CoT, $T=0.8$, top-p = 0.95) va ovoz berish. Oʻzbekcha soʻrovda kichik model koʻp xato qiladi; rus yoki ingliz tilidagi soʻrov bilan solishtiring. Katak faqat internetli Colab da ishlaydi.

In [ ]:
# haqiqiy modelda self-consistency: 7 ta sempling qilingan mulohaza
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if "llm" not in globals():
        name = "Qwen/Qwen2.5-0.5B-Instruct"
        tok = AutoTokenizer.from_pretrained(name)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = "auto" if device == "cuda" else torch.float32     # CPU da float32 bfloat16 dan tezroq
        llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)

    k = 4
    text = tok.apply_chat_template([{"role": "user", "content": make_prompt(problems[k], "zero-shot CoT")}],
                                   tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors="pt").to(llm.device)
    torch.manual_seed(0)
    out = llm.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.8, top_p=0.95,
                       num_return_sequences=7, pad_token_id=tok.eos_token_id)
    paths = [tok.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True) for o in out]
    preds = [extract_answer(t) for t in paths]
    for t, a in zip(paths, preds):
        print(f"[{a}] " + t.replace("\n", " ")[:120])
    votes = Counter(preds)
    print("\novozlar:", dict(votes), " ->", votes.most_common(1)[0][0], "  gold =", gold[k])
except Exception as e:
    print("Modelni ishga tushirib boʻlmadi (Colab da internet kerak):", e)

## 5. Topshiriqlar

1. **min-p.** min-p filtrini yozing: $p_i \ge \text{min\_p}\cdot \max_j p_j$ boʻlgan tokenlarni qoldirib, qayta normallang. 1-boʻlimdagi logitlarda $T = 1$ va $T = 2$ uchun min-p = 0.1 va top-p = 0.9 qoldirgan tokenlar sonini solishtiring. Qaysi filtr haroratga kuchliroq javob beradi?
2. **Takrorlash uchun jarima.** `generate` ga `penalty` parametrini qoʻshing: `out` da allaqachon bor soʻzlarning logitlaridan `penalty` ayiriladi. $T = 0.5$ uchun `penalty` = 0, 1, 3 da turli gaplar ulushini va gapning oʻrtacha uzunligini hisoblang.
3. **Oʻz soʻrovingiz.** Sharhlarni «ijobiy / salbiy» deb tasniflash uchun zero-shot va few-shot (3 ta misol) shablonlarini hamda erkin javobdan belgini ajratib oladigan `parse_label` funksiyasini yozing («Koʻproq ijobiy», «Belgi: salbiy.»). Uni oʻzingiz oʻylab topgan 4 ta javobda tekshiring.
4. **Nechta ovoz kerak?** Formula boʻyicha $p = 0.6$ va $p = 0.55$ uchun ovoz berish aniqligi ≥ 0.9 boʻladigan eng kichik toq $N$ ni toping. Nega $p = 0.5$ da bunday $N$ mavjud emas?
5. **Javobni tekshirish.** `check_steps` ni toʻldiring: «Javob:» dan keyingi son oxirgi qadam natijasiga mos kelmasa, u ham xato haqida xabar bersin. Barcha tayyorlab qoʻyilgan javoblarda va «2 · 15 = 30, 30 + 66 = 96. Javob: 69» satrida tekshiring.

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing